# 라이브러리

In [1]:
import csv
import hashlib
import io
import json
import math
import os
import random
import re
import shutil
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from PIL import Image

# 기본 설정
* 여기서 사용할 변수들 선언

In [6]:
# ai-server 폴더
try:
    # .py 파일일 때
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    # .ipynb 파일일 때 
    BASE_DIR = Path.cwd().resolve().parent

OUT_DIR = BASE_DIR / "data" / "car_images"
STAGE_DIR = OUT_DIR / "_download"  # 내려받는 중간 폴더. 마지막에 지운다

API = "https://commons.wikimedia.org/w/api.php"
# 위키미디어 API 규칙 (2026): User-Agent에 연락처(이메일이나 URL)가 있으면 1분에 200번,
# 없으면 같은 IP에서 1분에 10번까지만 부를 수 있다. 수업처럼 한 IP에서 여럿이 돌리면 더 느려진다.
# 연락처를 넣으려면 실행 전에:  set WIKI_CONTACT=me@example.com   (Mac: export WIKI_CONTACT=...)
CONTACT = os.getenv("WIKI_CONTACT", "stajun@naver.com").strip()
USER_AGENT = (f"usedcar-ai-edu/1.0 ({CONTACT or 'educational transfer-learning example'}) "
              f"python-urllib/{sys.version_info.major}.{sys.version_info.minor}")

THUMB_WIDTH = 330      # 위키미디어 표준 썸네일 크기 중 하나
PER_CLASS = 450        # 차종마다 최종으로 남길 사진 수 (최대)
MAX_LIST = 250         # 모델 하나에서 목록으로 볼 사진 수 (최대)
MAX_CATS = 15          # 모델 하나에서 둘러볼 카테고리 수 (최대)
MAX_DEPTH = 3          # 하위 카테고리를 몇 단계까지 내려갈지
CAR_PROB_MIN = 0.30    # ImageNet '자동차류' 확률 합이 이보다 작으면 자동차 사진이 아니라고 본다
SPLIT = {"test": 0.15, "val": 0.15}  # 나머지 70%가 train
SEED = 42

# 폴더 이름 → 화면에 나가는 차종 이름 (routers/car_type.py 의 LABELS 와 같은 순서)
FOLDER_LABEL = {"light": "경차", "sedan": "세단", "suv": "SUV"}

# 차종별 국산차 모델과 위키미디어 카테고리 이름 (같은 차의 수출명도 함께)
CAR_MODELS = {
    "light": {
        "Kia Ray": ["Kia Ray"],
        "Kia Morning": ["Kia Morning", "Kia Picanto"],
        "Chevrolet Spark": ["Chevrolet Spark", "Daewoo Matiz"],
        "Hyundai Casper": ["Hyundai Casper"],
        "Daewoo Tico": ["Daewoo Tico"],
        "Hyundai Atos": ["Hyundai Atos"],
    },
    "sedan": {
        "Hyundai Sonata": ["Hyundai Sonata"],
        "Hyundai Avante": ["Hyundai Avante", "Hyundai Elantra"],
        "Hyundai Grandeur": ["Hyundai Grandeur", "Hyundai Azera"],
        "Kia K5": ["Kia K5", "Kia Optima"],
        "Kia K3": ["Kia K3", "Kia Forte"],
        "Kia K7/K8": ["Kia K7", "Kia Cadenza", "Kia K8"],
        "Genesis G70": ["Genesis G70"],
        "Genesis G80": ["Genesis G80"],
        "Genesis G90": ["Genesis G90"],
        # 1960~70년대 말리부는 쿠페, 왜건이 많아서 국내에 팔린 세대(2011년~)만 쓴다
        "Chevrolet Malibu": ["Chevrolet Malibu (Epsilon II)", "Chevrolet Malibu (E2XX)"],
    },
    "suv": {
        "Hyundai Tucson": ["Hyundai Tucson"],
        "Hyundai Santa Fe": ["Hyundai Santa Fe"],
        "Hyundai Palisade": ["Hyundai Palisade"],
        "Hyundai Kona": ["Hyundai Kona"],
        "Kia Sorento": ["Kia Sorento"],
        "Kia Sportage": ["Kia Sportage"],
        "Kia Seltos": ["Kia Seltos"],
        "Kia Mohave": ["Kia Mohave"],
        "Genesis GV70": ["Genesis GV70"],
        "Genesis GV80": ["Genesis GV80"],
        "SsangYong Tivoli": ["SsangYong Tivoli"],
        "SsangYong Torres": ["SsangYong Torres"],
        "SsangYong Rexton": ["SsangYong Rexton"],
    },
}

# 제목에 이런 말이 있으면 차 전체 사진이 아닐 가능성이 크다 (모든 차종 공통)
COMMON_EXCLUDE = re.compile(
    r"\b(interiors?|inside|dashboard|cockpit|engines?|motor|seats?|steering|wheels?|rims?|tires?|tyres?"
    r"|badges?|logos?|emblems?|trunk|boot|head ?lamps?|head ?lights?|tail ?lamps?|tail ?lights?|lamps?|lights?"
    r"|details?|keys?|console|gauges?|instruments?|doors?|hood|bonnet|grilles?|mirrors?|exhaust|charging|plug"
    r"|scale ?models?|model ?cars?|toys?|miniatures?|die-?cast|lego|brochures?|advert\w*|posters?|drawings?"
    r"|sketch\w*|render\w*|concept|crash\w*|accidents?|wreck\w*|burn\w*|rally|rac(e|es|ing)|nascar|parts?"
    r"|pick-?ups?|trucks?|vans?|bus|ambulance|logos?)\b",
    re.IGNORECASE,
)
# 세단 카테고리에 섞여 있는 다른 모양(해치백, 쿠페, 왜건)
SEDAN_EXCLUDE = re.compile(
    r"\b(hatch\w*|coupes?|coupés?|koup|wagons?|sportswagon|estate|touring|convertibles?|cabrio\w*"
    r"|liftback|fastback|5-?doors?|forte ?5|gt)\b",
    re.IGNORECASE,
)

# ImageNet 1000종류 가운데 '자동차류'. 이 확률의 합으로 자동차 사진인지 본다
VEHICLE_CLASSES = [
    "ambulance", "beach wagon", "cab", "convertible", "grille", "jeep", "limousine", "minibus",
    "minivan", "Model T", "moving van", "pickup", "police van", "racer", "sports car", "tow truck",
]
# ---------- 위키미디어 API ----------
# API 요청 사이에 쉬는 시간(초). 한도보다 빨리 부르면 429(요청이 너무 많음)를 받는다
API_DELAY = 0.5 if CONTACT else 6.5
_last_api_call = 0.0

# 함수들

In [7]:
def http_get(url, timeout=30):
    """URL을 읽어 bytes로 돌려준다. 실패하면 조금 쉬었다가 다시 시도한다"""
    # 5번 시도. 실패하면 다시 시도
    for attempt in range(6):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
            with urllib.request.urlopen(req, timeout=timeout) as res:
                return res.read()
        except urllib.error.HTTPError as e:
            # 잘못된 URL은 재시도를 할 필요가 없음
            if e.code == 404:
                return None
            # 429면 서버가 알려 준 시간(Retry-After)만큼 기다린다
            retry_after = e.headers.get("Retry-After", "")
            # 서버에서 알려준 시간이 숫자이면 숫자로 변환, 아니면 지정된 시간을 계산
            wait = int(retry_after) if retry_after.isdigit() else 5 * (attempt + 1)
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            wait = 2 ** attempt
        # 60초 또는 서버가 알려준 시간 중 작은 시간에 맞춰 대기
        time.sleep(min(wait, 60))
    return None
# url 뒤에 ?a=b&c=d 형태로 만들어서 데이터를 요청하고 요청된 데이터를 json으로 변환
def api(params):
    # 여기서 사용하는 _last_api_call 변수는 전역변수 _last_api_call입니다
    global _last_api_call
    time.sleep(max(0.0, _last_api_call + API_DELAY - time.time()))
    _last_api_call = time.time()
    params = {**params, "format": "json", "formatversion": "2"}
    data = http_get(API + "?" + urllib.parse.urlencode(params))
    if data is None:
        raise RuntimeError("위키미디어 API에 연결하지 못했습니다")
    return json.loads(data)

def list_files(categories, exclude_res):
    """카테고리(와 하위 카테고리)에 있는 사진 파일 제목을 모은다"""
    files, seen = [], set()
    queue = [("Category:" + c, 0) for c in categories]
    while queue and len(seen) < MAX_CATS and len(files) < MAX_LIST:
        cat, depth = queue.pop(0)
        if cat in seen:
            continue
        seen.add(cat)
        cont = {}
        while True:
            # URL을 통해 dict를 가져옴
            data = api({"action": "query", "list": "categorymembers", "cmtitle": cat,
                        "cmtype": "file|subcat", "cmlimit": 500, **cont})
            # .get("query", {})에 {}를 넣는 이유. 다음 get에서 에러가 발생하지 않게하기위해
            for m in data.get("query", {}).get("categorymembers", []):
                title = m["title"]
                if any(r.search(title) for r in exclude_res):
                    continue
                if m["ns"] == 14 and depth < MAX_DEPTH:      # 하위 카테고리
                    queue.append((title, depth + 1))
                elif m["ns"] == 6 and title.lower().endswith((".jpg", ".jpeg", ".png")):
                    files.append(title)
            if "continue" not in data:
                break
            cont = data["continue"]
    return list(dict.fromkeys(files))  # 순서를 지키면서 중복 제거


def strip_html(text):
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", "", text or "")).strip()


def image_infos(titles):
    """파일 제목 → 썸네일 주소, 원본 페이지, 라이선스, 작성자"""
    infos = []
    for i in range(0, len(titles), 50):
        data = api({"action": "query", "titles": "|".join(titles[i:i + 50]), "prop": "imageinfo",
                    "iiprop": "url|mime|size|extmetadata", "iiurlwidth": THUMB_WIDTH,
                    "iiextmetadatafilter": "LicenseShortName|Artist"})
        for page in data.get("query", {}).get("pages", []):
            ii = (page.get("imageinfo") or [None])[0]
            if not ii or ii.get("mime") not in ("image/jpeg", "image/png"):
                continue
            w, h = ii.get("width", 0), ii.get("height", 0)
            if w < 300 or h < 200 or w / max(h, 1) > 2.5:  # 너무 작거나 파노라마
                continue
            meta = ii.get("extmetadata", {})
            infos.append({
                "title": page["title"],
                "thumb": ii.get("thumburl") or ii["url"],
                "page": ii.get("descriptionurl", ""),
                "license": meta.get("LicenseShortName", {}).get("value", ""),
                "artist": strip_html(meta.get("Artist", {}).get("value", ""))[:100],
            })
    return infos

# 이미지 다운로드 함수

In [9]:
# ---------- 1~3. 목록 모으기, 내려받기 ----------
def collect():
    rng = random.Random(SEED)
    records, used_titles = [], set()

    for folder, models in CAR_MODELS.items():
        per_model = math.ceil(PER_CLASS * 1.3 / len(models))  # 걸러질 것을 생각해서 넉넉히
        exclude = [COMMON_EXCLUDE] + ([SEDAN_EXCLUDE] if folder == "sedan" else [])
        print(f"\n[{FOLDER_LABEL[folder]}] 모델 {len(models)}개, 모델마다 최대 {per_model}장")

        for model_name, categories in models.items():
            titles = [t for t in list_files(categories, exclude) if t not in used_titles]
            rng.shuffle(titles)  # 이름순으로 앞에서만 자르면 같은 사람이 찍은 사진만 모인다
            infos = image_infos(titles[:per_model])
            used_titles.update(info["title"] for info in infos)
            for info in infos:
                info.update(folder=folder, car_model=model_name)
            records.extend(infos)
            print(f"  {model_name:<18} 목록 {len(titles):>4}장 → {len(infos):>3}장 받을 예정")

    print(f"\n내려받기: {len(records)}장")
    slug_count = defaultdict(int)
    for r in records:
        slug = re.sub(r"[^a-z0-9]+", "_", r["car_model"].lower()).strip("_")
        slug_count[slug] += 1
        r["file"] = f"{slug}_{slug_count[slug]:03d}.jpg"

    def fetch(r):
        data = http_get(r["thumb"])
        if data is None:
            return None
        try:
            image = Image.open(io.BytesIO(data))
            image.load()
            image = image.convert("RGB")
        except Exception:
            return None
        path = STAGE_DIR / r["folder"] / r["file"]
        path.parent.mkdir(parents=True, exist_ok=True)
        image.save(path, "JPEG", quality=92)
        r["path"] = path
        r["md5"] = hashlib.md5(data).hexdigest()
        return r

    # 동시에 2개까지만 받는다 (서버에 부담을 주지 않게)
    with ThreadPoolExecutor(max_workers=2) as pool:
        done = [r for r in pool.map(fetch, records) if r is not None]

    # 완전히 같은 사진 제거
    unique, seen_md5 = [], set()
    for r in done:
        if r["md5"] not in seen_md5:
            seen_md5.add(r["md5"])
            unique.append(r)
    print(f"받은 사진 {len(done)}장, 중복 제거 후 {len(unique)}장")
    return unique


# 사전학습 모델로 자동차가 아닌 사진들 거르는 함수

In [10]:
# ---------- 4. 사전학습 모델로 자동차가 아닌 사진 거르기 ----------
def car_filter(records):
    import torch
    from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

    weights = EfficientNet_B0_Weights.IMAGENET1K_V1
    model = efficientnet_b0(weights=weights).eval()
    preprocess = weights.transforms()
    categories = weights.meta["categories"]
    vehicle_idx = [categories.index(name) for name in VEHICLE_CLASSES]

    with torch.inference_mode():
        for i in range(0, len(records), 32):
            batch = records[i:i + 32]
            x = torch.stack([preprocess(Image.open(r["path"]).convert("RGB")) for r in batch])
            prob = model(x).softmax(dim=1)
            car_prob = prob[:, vehicle_idx].sum(dim=1)
            for r, p, top in zip(batch, car_prob, prob.argmax(dim=1)):
                r["car_prob"] = round(float(p), 3)
                r["imagenet_top1"] = categories[int(top)]

    keep = [r for r in records if r["car_prob"] >= CAR_PROB_MIN]
    rejected = [r for r in records if r["car_prob"] < CAR_PROB_MIN]
    print(f"자동차 사진 {len(keep)}장, 자동차가 아니라고 본 사진 {len(rejected)}장")
    return keep, rejected


# 기타 함수

In [11]:
# ---------- 5. train/val/test 나누기 ----------
def group_key(title):
    """'0 Kia Ray 1.jpg', '0 Kia Ray 2.jpg' 처럼 번호만 다른 사진은 같은 묶음으로 본다"""
    name = Path(title.split(":", 1)[-1]).stem.lower()
    name = re.sub(r"\(.*?\)", " ", name)
    name = re.sub(r"[\d_\-.,]+", " ", name)
    return re.sub(r"\s+", " ", name).strip()


def split(records):
    """차 모델마다 70/15/15로 나눈다. 차종 단위로만 나누면 한 모델이 시험에 몰릴 수 있다"""
    rng = random.Random(SEED)
    by_folder = defaultdict(list)
    for r in records:
        by_folder[r["folder"]].append(r)

    result = []
    for folder in FOLDER_LABEL:
        items = sorted(by_folder[folder], key=lambda r: r["title"])
        rng.shuffle(items)
        items = items[:PER_CLASS]
        by_model = defaultdict(list)
        for r in items:
            by_model[r["car_model"]].append(r)

        count = {"train": 0, "val": 0, "test": 0}
        for model_items in by_model.values():
            groups = defaultdict(list)
            for r in model_items:
                groups[group_key(r["title"])].append(r)
            group_list = list(groups.values())
            rng.shuffle(group_list)

            quota = {name: round(len(model_items) * ratio) for name, ratio in SPLIT.items()}
            filled = {"test": 0, "val": 0}
            for group in group_list:
                # 묶음째로 test → val 순서로 빈자리가 있는 곳에 넣고, 나머지는 train
                target = next((s for s in ("test", "val") if filled[s] + len(group) <= quota[s]), "train")
                if target != "train":
                    filled[target] += len(group)
                count[target] += len(group)
                for r in group:
                    r["split"] = target
        print(f"  {FOLDER_LABEL[folder]}: train {count['train']} / val {count['val']} / test {count['test']}")
        result += items
    return result


def move(records, rejected):
    for r in rejected:
        r["split"] = "rejected"
    for r in records + rejected:
        dest = OUT_DIR / r["split"] / r["folder"] / r["file"]
        if Path(r["path"]) != dest:
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(r["path"], dest)
        r["path"] = dest
    shutil.rmtree(STAGE_DIR, ignore_errors=True)

    with open(OUT_DIR / "credits.csv", "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(["path", "split", "car_type", "car_model", "car_prob", "imagenet_top1",
                    "title", "source", "license", "author"])
        for r in sorted(records + rejected, key=lambda r: str(r["path"])):
            w.writerow([r["path"].relative_to(OUT_DIR).as_posix(), r["split"], FOLDER_LABEL[r["folder"]],
                        r["car_model"], r["car_prob"], r["imagenet_top1"], r["title"], r["page"],
                        r["license"], r["artist"]])


def resplit():
    """이미 받은 사진을 credits.csv를 보고 train/val/test로 다시 나눈다 (내려받지 않는다)"""
    folder_of = {label: folder for folder, label in FOLDER_LABEL.items()}
    records, rejected = [], []
    with open(OUT_DIR / "credits.csv", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            r = {"folder": folder_of[row["car_type"]], "car_model": row["car_model"],
                 "title": row["title"], "page": row["source"], "license": row["license"],
                 "artist": row["author"], "car_prob": row["car_prob"],
                 "imagenet_top1": row["imagenet_top1"], "path": OUT_DIR / row["path"],
                 "file": Path(row["path"]).name}
            (rejected if row["split"] == "rejected" else records).append(r)
    print("train / val / test 다시 나누기")
    move(split(records), rejected)
    print(f"완료: {OUT_DIR}")


# 이미지 다운로드 실행하는 함수

In [12]:
def main():
    if "--resplit" in sys.argv:
        resplit()
        return
    if (OUT_DIR / "train").exists():
        print(f"이미 사진이 있습니다: {OUT_DIR}\n다시 모으려면 이 폴더를 지우고 실행하세요"
              "\n(나누기만 다시 하려면: python models/collect_car_images.py --resplit)")
        return
    shutil.rmtree(STAGE_DIR, ignore_errors=True)

    if CONTACT:
        print(f"User-Agent 연락처: {CONTACT} (API 1분에 200번까지)")
    else:
        print("User-Agent에 연락처가 없어서 API를 1분에 10번만 부른다 (30~60분 걸린다)."
              " 빠르게 하려면 WIKI_CONTACT 환경 변수에 이메일을 넣는다")
    print("1~3. 위키미디어 공용에서 사진 모으기")
    records = collect()
    print("\n4. 사전학습 모델(EfficientNet-B0, ImageNet)로 자동차가 아닌 사진 거르기")
    keep, rejected = car_filter(records)
    print("\n5. train / val / test 나누기")
    keep = split(keep)
    move(keep, rejected)
    print(f"\n완료: {OUT_DIR}")

In [13]:
main()

User-Agent 연락처: stajun@naver.com (API 1분에 200번까지)
1~3. 위키미디어 공용에서 사진 모으기

[경차] 모델 6개, 모델마다 최대 98장
  Kia Ray            목록  139장 →  98장 받을 예정
  Kia Morning        목록  260장 →  98장 받을 예정
  Chevrolet Spark    목록  259장 →  98장 받을 예정
  Hyundai Casper     목록  656장 →  98장 받을 예정
  Daewoo Tico        목록   87장 →  87장 받을 예정
  Hyundai Atos       목록  206장 →  98장 받을 예정

[세단] 모델 10개, 모델마다 최대 59장
  Hyundai Sonata     목록  516장 →  59장 받을 예정
  Hyundai Avante     목록  333장 →  58장 받을 예정
  Hyundai Grandeur   목록  268장 →  59장 받을 예정
  Kia K5             목록  437장 →  59장 받을 예정
  Kia K3             목록  321장 →  59장 받을 예정
  Kia K7/K8          목록  281장 →  59장 받을 예정
  Genesis G70        목록  287장 →  59장 받을 예정
  Genesis G80        목록 1693장 →  59장 받을 예정
  Genesis G90        목록 2097장 →  59장 받을 예정
  Chevrolet Malibu   목록  249장 →  59장 받을 예정

[SUV] 모델 13개, 모델마다 최대 45장
  Hyundai Tucson     목록  319장 →  44장 받을 예정
  Hyundai Santa Fe   목록  371장 →  45장 받을 예정
  Hyundai Palisade   목록  623장 →  45장 받을 예정
  Hyundai Kona       목록  294장 → 